# Architecture 1 — three-seed exploratory run: 42 → 123 → 2026 (research record §40 step 6, §46, §51, §52)

**Status of this run.** C2 FAILED its pre-registered criterion (§50). This is a low-prior exploratory / falsification test,
not a confirmatory experiment and not a superiority experiment. **No EMA** (P protocol has none; explicit deviation from the §40 note).
The five checks are per-seed closure criteria against P-42. Nothing is tuned between seeds.

GPU runtime. **Cell 5 trains** the three seeds one after another, each in its own run directory, each from a fresh
initialisation. After a disconnect rerun cells 1–5: completed seeds are kept, the interrupted seed resumes from its own
checkpoint, later seeds start fresh. The 3-seed summary is written only when all three seeds have finished.
P protocol reused exactly (batch 2, AdamW 1e-4 / wd 0.05, weighted CORN, ≤ 50 epochs, val-QWK early stopping).
Refuses to run on a missing/mismatched model SHA, an incomplete or legacy cache, failed Stage-3/RGB parity, an inconsistent
bundle fingerprint, the wrong APTOS split, or non-APTOS (e.g. IDRiD) ids.

In [ ]:
# ==== [1] SETUP ====
import os, subprocess, sys
REPO_URL = "https://github.com/yasodharan27/diabetic_retinoplasty.git"
REPO_DIR = "/content/diabetic_retinoplasty"
BRANCH = "main"
if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "origin", BRANCH], check=True)
for _p in (REPO_DIR, os.path.join(REPO_DIR, "colab", "common")):
    if _p not in sys.path:
        sys.path.insert(0, _p)
REPO_COMMIT = subprocess.run(["git", "-C", REPO_DIR, "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
import setup as colab_setup
SETUP_INFO = colab_setup.setup()          # Drive, repo, requirements, env vars. No TJDR / IDRiD / Stage-4 weights needed.
import stage4_v2_setup                    # only for stage_bundle (remount-safe Drive -> local copy)
import glob, json, posixpath
import numpy as np, pandas as pd, tensorflow as tf
import colab_config
import arch1_data as ad, arch1_model as a1, arch1_train as at
import multiseed_runs as msr, pl_convnext as pl, weighted_corn
import pipeline_v2_config as v2cfg, stage34_cache_v2 as cache
assert tf.config.list_physical_devices("GPU"), "GPU runtime required"
print("repo", REPO_COMMIT)

In [ ]:
# ==== [2] INPUTS: Stage-4 model identity, bundle, protocol, C2 decision, seeds ====
EXP = colab_config.DRIVE.experiments_root
EXPECTED_MODEL_SHA = "cb5fc7a8d370af7d2ae191cadaebdde858f71820d147fb76f852b757766f8ad8"   # gate-passed Stage-4 v2 (§47/§48)
STAGE4_RUN_DIR = posixpath.join(EXP, "Stage4V2", "stage4v2_k4_seed42_bs8")
MODEL_SHA = json.load(open(posixpath.join(STAGE4_RUN_DIR, "exported_model.json")))["model_sha256"]
assert MODEL_SHA == EXPECTED_MODEL_SHA, f"exported Stage-4 model {MODEL_SHA} is not the approved {EXPECTED_MODEL_SHA}"
GEN4 = cache.stage4_generation_id(MODEL_SHA, len(v2cfg.STAGE4_V2A_CLASSES))
assert GEN4 == "s4v2-cb5fc7a8d370-K4", GEN4
BUNDLE_ID = cache.bundle_id(v2cfg.STAGE2_RGB_GENERATION, v2cfg.STAGE3_GENERATION, GEN4)
# C2 (§50) is read ONLY to record its decision; nothing in it is a training target or a hyper-parameter.
C2_RESULTS = posixpath.join(EXP, "C2", "c2_cb5fc7a8d370_2026-10-02_08-46-52", "results.json")
C2_RESULTS_SHA256 = "2fb64280047df317cb182e53f0c40f14c938f5e97a13959bf69475dbc9fab13b"
ACKNOWLEDGE_C2_FAIL = True          # C2 FAILED: this run is a low-prior exploratory / falsification test (§40, §51)
assert os.path.exists(C2_RESULTS), f"C2 results not found: {C2_RESULTS}"
assert cache.sha256_file(C2_RESULTS) == C2_RESULTS_SHA256, "C2 results.json is not the recorded §50 file"
C2_PASS = bool(json.load(open(C2_RESULTS))["decision"]["PASS"])
assert C2_PASS or ACKNOWLEDGE_C2_FAIL, "C2 FAILED -- set ACKNOWLEDGE_C2_FAIL only for a low-prior test"
SIX_RUN_DIR = msr.experiment_root(posixpath.join(EXP, "ImprovedTraining"), "improved_multiseed_2026_09")
PREREG, _ = msr.load_and_verify_preregistration(posixpath.join(SIX_RUN_DIR, msr.PREREGISTRATION_FILENAME))
CLASS_WEIGHTS = list(PREREG["class_weights"])
at.assert_protocol(PREREG, CLASS_WEIGHTS)
SEEDS = at.SEEDS
assert SEEDS == (42, 123, 2026) and at.EMA == "none"
RUN_DIRS = {s: at.run_dir_for(EXP, MODEL_SHA, s) for s in SEEDS}
SUMMARY_DIR = at.summary_dir_for(EXP, MODEL_SHA)
print("model", MODEL_SHA, "| bundle", BUNDLE_ID)
print("C2:", "PASS" if C2_PASS else "FAIL -- acknowledged; exploratory low-prior run", "| EMA:", at.EMA)
for _s in SEEDS:
    print(f"  seed {_s}: {at.seed_state(RUN_DIRS[_s]):<11} {RUN_DIRS[_s]}")

In [ ]:
# ==== [3] VERIFY THE BUNDLE ON DRIVE, STAGE IT LOCALLY (~14.5 GB), VERIFY AGAIN ====
DRIVE_ROOTS = dict(ad.DEFAULT_ROOTS)
LOCAL_ROOTS = {k: posixpath.join("/content/cache_v2", os.path.basename(v)) for k, v in DRIVE_ROOTS.items()}
ad.Arch1Bundle(expected_bundle_id=BUNDLE_ID, expected_split_sha256=v2cfg.SPLIT_SHA256, expected_stage4_sha256=MODEL_SHA)
stage4_v2_setup.stage_bundle(BUNDLE_ID, DRIVE_ROOTS, LOCAL_ROOTS)
BUNDLE = ad.Arch1Bundle(expected_bundle_id=BUNDLE_ID, expected_split_sha256=v2cfg.SPLIT_SHA256,
                        expected_stage4_sha256=MODEL_SHA, roots=LOCAL_ROOTS)
BUNDLE.require(BUNDLE.population)
# Identity: approved Stage-3 / Stage-4 models, and train/val exactly the authoritative split minus the pinned empty-FOV ids.
assert BUNDLE.stage3_sha256 == v2cfg.STAGE3_LWNET_SHA256 and BUNDLE.stage4_sha256 == EXPECTED_MODEL_SHA
assert BUNDLE.stage4_generation == GEN4 and BUNDLE.bundle["stage3_generation"] == v2cfg.STAGE3_GENERATION
_tr, _va, _split_sha = msr.verify_split()
_empty = set(v2cfg.APTOS_EMPTY_FOV_IDS)
assert _split_sha == v2cfg.SPLIT_SHA256
assert list(BUNDLE.train_ids) == [str(i) for i, _ in _tr if str(i) not in _empty] and len(BUNDLE.train_ids) == 2921
assert list(BUNDLE.val_ids) == [str(i) for i, _ in _va if str(i) not in _empty] and len(BUNDLE.val_ids) == 730
print(json.dumps(BUNDLE.describe(), indent=1))
print("train", len(BUNDLE.train_ids), "| val", len(BUNDLE.val_ids), "| split and population verified")

In [ ]:
# ==== [4] PINNED CONVNEXT WEIGHTS + PRE-FLIGHT (zero-initialised model == P on real samples) + P-42 REFERENCE ====
WEIGHTS = pl.ensure_pretrained_weights(posixpath.join(EXP, "PL_ConvNeXtPriors", "pretrained_weights"))
REFERENCE, REFERENCE_ARRAYS = pl.load_reference(WEIGHTS["local_path"])
model = at.build_compiled_model(BUNDLE.channels, SEEDS[0], REFERENCE, CLASS_WEIGHTS, mixed_precision=False)
p_model = pl.build_pl_model("P", SEEDS[0], REFERENCE_ARRAYS)
samples = [BUNDLE.load_sample(i) for i in BUNDLE.val_ids[:2]]
x = {k: np.stack([s[k] for s in samples]) for k in ("rgb", "vessel", "pathology")}
s5 = np.concatenate([x["rgb"], np.zeros(x["rgb"].shape[:3] + (5,), "float32")], -1)
p_logits = p_model.predict([s5, np.zeros((2, 256, 256, 3), "float32"), np.zeros((2, 1), "float32")], verbose=0)
diff = float(np.max(np.abs(model.predict_on_batch(x) - p_logits)))
assert diff <= pl.PL_P_EQUIVALENCE_TOL, f"zero-initialised Architecture 1 != P ({diff})"
print("pre-flight OK: |Arch1_init - P| =", diff, "| parameters", a1.parameter_report(model))
del model, p_model
# The reference for the five per-seed checks (all three seeds): P-42 BEST, on the same 730 validation images, same order.
P42 = sorted(glob.glob(posixpath.join(EXP, "PL_ConvNeXtPriors", "*", "P_convnext_rgb", "seed_42", "evaluation", "metrics_best.json")))
assert len(P42) == 1, P42
P42_METRICS = json.load(open(P42[0]))
_p42_ids = pd.read_csv(posixpath.join(os.path.dirname(P42[0]), "per_sample_best.csv"), dtype={"image_id": str})["image_id"]
assert list(_p42_ids) == list(BUNDLE.val_ids), "P-42 was evaluated on a different validation set"
SOURCES = {"p42_metrics": P42[0], "p42_metrics_sha256": cache.sha256_file(P42[0]),
           "c2_results": C2_RESULTS, "c2_results_sha256": C2_RESULTS_SHA256, "repo_commit": REPO_COMMIT}
print("P-42:", {k: round(P42_METRICS[k], 4) for k in ("qwk", "auroc_ge3_g4_vs_g012", "grade3_recall", "false_urgent_rate")})

In [ ]:
# ==== [5] TRAIN + EVALUATE SEEDS 42 -> 123 -> 2026 (resumable) -- THIS TRAINS ====
# Per seed: fresh model (pretrained ConvNeXt + that seed's initialisation) -> train -> BEST/LAST evaluation ->
# lesion shuffle -> five checks vs P-42 -> verdict.json / verdict.md in that seed's own directory.
# Any error stops the sequence here (no seed is skipped). Rerun cells 1-5 to continue: completed seeds are kept,
# the interrupted seed resumes from its own checkpoint. The 3-seed summary is written only after all three.
VERDICTS, SUMMARY = at.run_sequence(EXP, BUNDLE, REFERENCE, CLASS_WEIGHTS, P42_METRICS, C2_PASS, repo_dir=REPO_DIR,
                                    staging_root="/content/arch1_staging", sources=SOURCES)

In [ ]:
# ==== [6] REPORT: 3-seed summary + per-seed verdicts (reads what cell 5 wrote; trains nothing) ====
assert all(at.seed_state(RUN_DIRS[s]) == "complete" for s in SEEDS), {s: at.seed_state(RUN_DIRS[s]) for s in SEEDS}
assert os.path.exists(posixpath.join(SUMMARY_DIR, "summary.json")), "summary not written -- rerun cell 5"
print(open(posixpath.join(SUMMARY_DIR, "summary.md"), encoding="utf-8").read())
for _s in SEEDS:
    print(open(posixpath.join(RUN_DIRS[_s], "verdict.md"), encoding="utf-8").read())
print("summary:", SUMMARY_DIR)